In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class NaiveBayes:
    def __init__(self):
        self.priors = {}  # 先验概率
        self.class_priors = {} # 类条件概率
        self.mean = {} # 连续特征的均值
        self.var = {} # 连续特征的方差

    def calculate_num(self,data):
        """统计类别数目"""
        class_count = {}
        for i in data:
            if i not in class_count.keys():
                class_count[i] = 0
            class_count[i] += 1
        return class_count

    def fit(self, X, y):
        """训练模型"""
        samples_num, features_num = X.shape
        labels_count = self.calculate_num(y)
        # 计算先验概率
        for label in labels_count.keys():
            Dc = labels_count[label]
            D = len(y)    
            N = len(labels_count.keys())
            # 公式(7.19)
            prior = (Dc + 1) / float(D + N)
            self.priors[label] = prior
        # 计算类条件概率
        for label in labels_count.keys():
            for i in range(features_num):
                # 计算特征i在类别label下的条件概率
                feature_values = X[y == label, i]
                # 连续特征，假设符合高斯分布
                if type(feature_values[0]).__name__ == 'float':
                    # 计算均值和方差
                    mean = np.mean(feature_values)
                    var = np.var(feature_values)
                    self.mean[(label,i)] = mean
                    self.var[(label,i)] = var
                # 离散特征
                else:
                    feature_count = self.calculate_num(feature_values)
                    # 第i个属性可能的取值数
                    Ni = len(set(X[:,i]))
                    # 类别label的样本数
                    Dc = labels_count[label]
                    # 遍历当前特征的所有取值
                    for value in feature_count.keys():
                        Dcx = feature_count[value]
                        # 公式(7.20)
                        class_prior = (Dcx + 1) / (Dc + Ni)
                        self.class_priors[(label,i,value)] = class_prior

    def predict(self, X_test):
        """预测"""
        y_scores = []
        y_pred = []
        for sample in X_test:
            for label in self.priors.keys():
                score = self.priors[label]
                print('P(c = %s) = %.3f'%(label,self.priors[label]))
                # 遍历每一个特征
                for i in range(len(sample)):
                    # 连续特征
                    if type(sample[i]).__name__ == 'float':
                        mean = self.mean[(label,i)]
                        var = self.var[(label,i)]
                        # 公式(7.18)
                        class_prior = np.exp(-(sample[i] - mean)**2 / (2 * var)) / np.sqrt(2 * np.pi * var)
                        print('P(x_%d | c = %s) = %.3f'%(i+1,label,class_prior))
                        score *= class_prior
                    # 离散特征
                    else:
                        if (label,i,sample[i]) in self.class_priors.keys():
                            class_prior = self.class_priors[(label,i,sample[i])]
                        else:
                            Dc = self.calculate_num(y)[label]
                            Ni = len(set(X[:,i]))
                            class_prior = (0 + 1) / (Dc + Ni)
                        print('P(x_%d = %s | c = %s) = %.3f'%(i+1,sample[i],label,class_prior))
                        score *= class_prior
                print('P(c = %s | x) = %f'%(label,score))
                print('----------------------------------')
                y_scores.append(score)
            # 取出分数最高的类别作为预测结果
            index = np.argmax(y_scores)
            y_pred.append(list(self.priors.keys())[index])
        return y_pred

if __name__ == '__main__':
    # 加载数据集
    data = pd.read_csv('../Data/watermelon3.0.csv',encoding='ansi')
    X = data.iloc[:,1:-1].values
    y = data.iloc[:,-1].values
    # 训练
    NB = NaiveBayes()
    NB.fit(X,y)
    # 测试
    X_test = ['青绿','蜷缩','清脆','清晰','凹陷','硬滑',0.697,0.46]
    y_pred = NB.predict([X_test])
    print('The predicted class is:',y_pred)
   


P(c = 是) = 0.474
P(x_1 = 青绿 | c = 是) = 0.364
P(x_2 = 蜷缩 | c = 是) = 0.545
P(x_3 = 清脆 | c = 是) = 0.091
P(x_4 = 清晰 | c = 是) = 0.727
P(x_5 = 凹陷 | c = 是) = 0.545
P(x_6 = 硬滑 | c = 是) = 0.700
P(x_7 | c = 是) = 1.962
P(x_8 | c = 是) = 0.669
P(c = 是 | x) = 0.003114
----------------------------------
P(c = 否) = 0.526
P(x_1 = 青绿 | c = 否) = 0.333
P(x_2 = 蜷缩 | c = 否) = 0.333
P(x_3 = 清脆 | c = 否) = 0.250
P(x_4 = 清晰 | c = 否) = 0.250
P(x_5 = 凹陷 | c = 否) = 0.250
P(x_6 = 硬滑 | c = 否) = 0.636
P(x_7 | c = 否) = 1.194
P(x_8 | c = 否) = 0.042
P(c = 否 | x) = 0.000029
----------------------------------
The predicted class is: ['是']
